In [ ]:
"""
setup_data.py - 데이터 압축 해제 스크립트
Mac에서 만든 zip의 한글 파일명도 정상 처리
"""
import zipfile
from pathlib import Path
import shutil
import sys

def extract_with_encoding(zip_path, target):
    """Mac에서 만든 zip의 한글 파일명 깨짐 문제 해결"""
    with zipfile.ZipFile(zip_path, 'r') as z:
        for info in z.infolist():
            # __MACOSX 폴더 스킵
            if info.filename.startswith('__MACOSX'):
                continue
            
            # 파일명 인코딩 복구 시도 (cp437 → utf-8)
            try:
                filename = info.filename.encode('cp437').decode('utf-8')
            except (UnicodeDecodeError, UnicodeEncodeError):
                filename = info.filename
            
            # 디렉토리면 생성만
            if info.is_dir():
                (target / filename).mkdir(parents=True, exist_ok=True)
                continue
            
            # 파일 저장
            out_path = target / filename
            out_path.parent.mkdir(parents=True, exist_ok=True)
            with z.open(info) as src, open(out_path, 'wb') as dst:
                shutil.copyfileobj(src, dst)

def main():
    RAW_DIR = Path('../data/raw')
    EXTRACT_DIR = Path('../data/extracted')
    
    if not RAW_DIR.exists():
        print(f'❌ {RAW_DIR} 폴더가 없습니다.')
        sys.exit(1)
    
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    
    zip_files = sorted(RAW_DIR.glob('*.zip'))
    if not zip_files:
        print(f'⚠️  {RAW_DIR} 에 zip 파일이 없습니다.')
        sys.exit(1)
    
    print(f'📦 {len(zip_files)}개 zip 파일 처리 시작\n')
    
    extracted = 0
    skipped = 0
    
    for zip_path in zip_files:
        target = EXTRACT_DIR / zip_path.stem
        
        if target.exists() and any(target.iterdir()):
            print(f'  ⏭  {zip_path.name} (이미 풀림)')
            skipped += 1
            continue
        
        print(f'  📂 {zip_path.name}')
        target.mkdir(exist_ok=True)
        try:
            extract_with_encoding(zip_path, target)
            csv_count = len(list(target.glob('*.csv')))
            print(f'      ✅ CSV {csv_count}개 생성')
            extracted += 1
        except Exception as e:
            print(f'      ❌ 에러: {e}')
    
    print(f'\n{"="*50}')
    print(f'✅ 완료: 새로 해제 {extracted}개, 스킵 {skipped}개')
    print(f'📁 데이터 위치: {EXTRACT_DIR.resolve()}')

if __name__ == '__main__':
    main()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# seaborn 스타일을 먼저 적용 (이게 폰트를 덮어쓸 수 있어서 순서 중요!)
sns.set_style('whitegrid')

# 그 다음 한글 폰트 (Windows)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 출력 옵션
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:,.2f}'.format)

print(' 환경 설정 완료')



In [ ]:
import glob
import pandas as pd
from pathlib import Path

# 프로젝트 루트 자동 감지
ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent

# 모든 추정매출 파일 한 번에
sales_files = sorted(glob.glob(str(ROOT / 'data' / 'extracted' / '**' / '*추정매출*.csv'), recursive=True))

print(f'📂 발견된 파일:')
for f in sales_files:
    print(f'  - {Path(f).name}')

# 통합
sales = pd.concat([
    pd.read_csv(f, encoding='cp949') for f in sales_files
], ignore_index=True)

print(f'✅ 통합: {sales.shape}')

In [ ]:
import glob
import os
DATA_DIR = '../data/extracted/서울시_상권분석서비스(추정매출+영역)/'

# 추정매출 파일 자동 탐색
pattern = DATA_DIR + '*추정매출*.csv'
sales_files = sorted(glob.glob(pattern))

print(f'총 파일: {len(sales_files)}개')
for f in sales_files:
    print(f'  {os.path.basename(f)}')

# 통합
sales_list = []
for path in sales_files:
    df = pd.read_csv(path, encoding='cp949', low_memory=False)
    sales_list.append(df)

sales = pd.concat(sales_list, ignore_index=True)
print(f'\n📦 통합 완료: {len(sales):,}행 × {len(sales.columns)}컬럼')
print(f' 메모리: {sales.memory_usage(deep=True).sum()/1024**2:.1f} MB')

# 영역 파일도 자동 탐색
area_path = glob.glob(DATA_DIR + '*영역*상권*.csv')[0]
area = pd.read_csv(area_path, encoding='cp949')
print(f'\n  영역: {len(area):,}개 상권')

In [ ]:
# 데이터 모양과 타입
print(f'추정매출: {sales.shape}')
print(f'영역:    {area.shape}')

key_cols = ['기준_년분기_코드','상권_구분_코드_명','상권_코드','상권_코드_명',
            '서비스_업종_코드','서비스_업종_코드_명','당월_매출_금액','당월_매출_건수']
print(sales[key_cols].dtypes)

In [ ]:
최신_분기_코드 = sales['기준_년분기_코드'].max()
print(f'최신 분기: {최신_분기_코드}')

In [ ]:
# 시간/공간 범위
print(f' 분기 범위: {sales["기준_년분기_코드"].min()} ~ {sales["기준_년분기_코드"].max()}')
print(f' 분기 수:   {sales["기준_년분기_코드"].nunique()}개')
print(f'\n 상권 수:   {sales["상권_코드"].nunique():,}개')
print(f'  업종 수:   {sales["서비스_업종_코드"].nunique()}개')

# 상권 구분 분포
print(f'\n 상권 구분별 ( {최신_분기_코드} 분기 기준)')
print(sales[sales['기준_년분기_코드']==최신_분기_코드]
      .drop_duplicates('상권_코드')['상권_구분_코드_명'].value_counts())

In [ ]:
# 55개 컬럼을 카테고리별로 정리
매출금액_컬럼 = [c for c in sales.columns if '매출_금액' in c]
매출건수_컬럼 = [c for c in sales.columns if '매출_건수' in c]

요일_금액 = [c for c in 매출금액_컬럼 if any(d in c for d in ['월요일','화요일','수요일','목요일','금요일','토요일','일요일','주중','주말'])]
시간_금액 = [c for c in 매출금액_컬럼 if '시간대' in c]
성별_금액 = [c for c in 매출금액_컬럼 if '남성' in c or '여성' in c]
연령_금액 = [c for c in 매출금액_컬럼 if '연령대' in c]

print(f' 매출금액 컬럼: {len(매출금액_컬럼)}개')
print(f'   - 기본:    당월_매출_금액')
print(f'   - 요일별:  {len(요일_금액)}개 (월~일 + 주중/주말)')
print(f'   - 시간대:  {len(시간_금액)}개 (6구간)')
print(f'   - 성별:    {len(성별_금액)}개')
print(f'   - 연령대:  {len(연령_금액)}개 (10~60+)')
print(f'\n 매출건수 컬럼: {len(매출건수_컬럼)}개 (금액과 동일 구조)')

In [ ]:
# 객단가 컬럼 추가
sales['객단가'] = sales['당월_매출_금액'] / sales['당월_매출_건수'].replace(0, np.nan)

# 핵심 변수 기초통계
sales[['당월_매출_금액', '당월_매출_건수', '객단가']].describe(
    percentiles=[.01, .25, .5, .75, .95, .99]
)

In [ ]:
# 영역 정보 merge
매출_영역 = sales.merge(
    area[['상권_코드','자치구_코드_명','행정동_코드_명','엑스좌표_값','와이좌표_값']],
    on='상권_코드', how='left'
)
print(f'merge 후: {매출_영역.shape}')
print(f'결측 자치구: {매출_영역["자치구_코드_명"].isna().sum()}')

In [ ]:
# 최신 분기로 필터 후 상권 단위 집계
최신분기 = 매출_영역[매출_영역['기준_년분기_코드']==최신_분기_코드]

상권집계 = 최신분기.groupby(
    ['상권_코드','상권_코드_명','상권_구분_코드_명','자치구_코드_명']
).agg(
    총매출=('당월_매출_금액','sum'),
    총건수=('당월_매출_건수','sum'),
    업종수=('서비스_업종_코드','nunique'),
).reset_index()

상권집계['객단가'] = 상권집계['총매출'] / 상권집계['총건수']
상권집계['log_매출'] = np.log10(상권집계['총매출'].clip(lower=1))
상권집계['총매출_억'] = (상권집계['총매출']/1e8).round(1)

print(f'분석 대상 상권: {len(상권집계):,}개')
상권집계.head(5)

In [ ]:
# 전ㄴ체연도 로 필터 후 상권 단위 집계

전체_상권집계 = 매출_영역.groupby(
    ['상권_코드','상권_코드_명','상권_구분_코드_명','자치구_코드_명']
).agg(
    총매출=('당월_매출_금액','sum'),
    총건수=('당월_매출_건수','sum'),
    업종수=('서비스_업종_코드','nunique'),
).reset_index()

전체_상권집계['객단가'] = 전체_상권집계['총매출'] / 전체_상권집계['총건수']
전체_상권집계['log_매출'] = np.log10(전체_상권집계['총매출'].clip(lower=1))
전체_상권집계['총매출_억'] = (전체_상권집계['총매출']/1e8).round(1)

print(f'분석 대상 상권: {len(전체_상권집계):,}개')
전체_상권집계.head(5)

---

## 자치구별 매출 * 매출건수 * 객단가

목표:
- 현재 분석에서 나아가 자치구별 어느상권의 매출이 왜 높은지 심화분석.
- 객단가 -> 고가소비중심의 프리미엄 상권
-  매출건수 -> 유동 인구 중심의 상권

---
* 최근 년분기 기준의 자치구별 매출 구조 통계

In [ ]:

import matplotlib.pyplot as plt
import platform

# 한글 폰트
if platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 자치구별 집계
구분통계 = 상권집계.groupby('자치구_코드_명').agg(
    매출건수=('총건수', 'sum'),
    평균매출=('총매출', 'mean'),
    중앙매출=('총매출', 'median'),
    평균객단가=('객단가', 'mean')
).reset_index()

구분통계['평균객단가'] = 구분통계['평균객단가'].round(0)

# 평균매출 기준 정렬
구분통계 = 구분통계.sort_values('평균매출', ascending=True)

fig, axes = plt.subplots(1, 4, figsize=(22, 10))

# 1. 매출건수
axes[0].barh(구분통계['자치구_코드_명'], 구분통계['매출건수'], color='steelblue')
axes[0].set_title('자치구별 매출건수')
axes[0].set_xlabel('건수')

# 2. 평균매출
axes[1].barh(구분통계['자치구_코드_명'], 구분통계['평균매출'] / 1e8, color='coral')
axes[1].set_title('자치구별 평균매출 (억원)')
axes[1].set_xlabel('억원')

# 3. 중앙매출
axes[2].barh(구분통계['자치구_코드_명'], 구분통계['중앙매출'] / 1e8, color='mediumpurple')
axes[2].set_title('자치구별 중앙매출 (억원)')
axes[2].set_xlabel('억원')

# 4. 평균객단가
axes[3].barh(구분통계['자치구_코드_명'], 구분통계['평균객단가'], color='seagreen')
axes[3].set_title('자치구별 평균객단가 (원)')
axes[3].set_xlabel('원')

plt.tight_layout()
plt.show()

- 인사이트:
  - 1.중구와 송파구 "규모의 시장"
    매출건수 + 평균매출 -> 1~2위 .
    거래도 많은데 단가도 높다.
    ==> 유동인구 기반의 대형상권으로 볼 수 있음.

  - 2. 서초와 강남 "고단가 저빈도 소비"
    평균객단가는 압도적 1~2위 인데, 매출건수는 중구 와 송파보다 현저히 낮음.
    ==> 소수 고가 거래 중심의 시장구조. 대중적 접근성보다 프리미엄 소비층이 타겟.

  - 3. 노원구  "일반적이지 않는 상권구조"
    매출건수+평균매출 모두 중하위권인데, 중앙매출이 유독 높음. / 평균은 낮지만 중앙값이 높다는건 극단적 고매출 업체가 없는대신,=>중간 규모 업체들이 고르게 분포한다는 의미. 상권구조가 다른구들과 다르다.

  - 4. 중구와 송파  "평균과 중앙의 괴리가 큼"
   평균매출은 압도적 1위인데 중앙매출은 그에 비해 훨씬 낮음
   => 소수의 초대형 매출업체가 평균을 끌어올리는 구조. 상권내 양극화가 심함.

  - 5. 객단가 하위권 구들의 공통점  "생활밀착 상권"
   노원,도봉,은평 등 객단가, 평균매출 모두 하위권이지만 매출건수가 낮지 않음.
   => 저단가 고빈도 소비의 구조. 생활밀착형 상권임.


In [ ]:

import matplotlib.pyplot as plt
import platform

# 한글 폰트
if platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 상권유형별 집계
구분통계 = 상권집계.groupby('상권_구분_코드_명').agg(
    매출건수=('총건수', 'sum'),
    평균매출=('총매출', 'mean'),
    중앙매출=('총매출', 'median'),
    평균객단가=('객단가', 'mean')
).reset_index()

구분통계['평균객단가'] = 구분통계['평균객단가'].round(0)

# 평균매출 기준 정렬
구분통계 = 구분통계.sort_values('평균매출', ascending=True)

fig, axes = plt.subplots(1, 4, figsize=(22, 10))

# 1. 매출건수
axes[0].barh(구분통계['상권_구분_코드_명'], 구분통계['매출건수'], color='steelblue')
axes[0].set_title('상권유형별 매출건수')
axes[0].set_xlabel('건수')


# 2. 평균매출
axes[1].barh(구분통계['상권_구분_코드_명'], 구분통계['평균매출'] / 1e8, color='coral')
axes[1].set_title('상권유형별 평균매출 (억원)')
axes[1].set_xlabel('억원')

# 3. 중앙매출
axes[2].barh(구분통계['상권_구분_코드_명'], 구분통계['중앙매출'] / 1e8, color='mediumpurple')
axes[2].set_title('상권유형별 중앙매출 (억원)')
axes[2].set_xlabel('억원')

# 4. 평균객단가
axes[3].barh(구분통계['상권_구분_코드_명'], 구분통계['평균객단가'], color='seagreen')
axes[3].set_title('상권유형별 평균객단가 (원)')
axes[3].set_xlabel('원')

plt.tight_layout()
plt.show()

* 인사이트:
- 1. 관광특구 
   << 방문수보다 매출규모 중심상권 구조 >>
   -  평균 매출 1위.
   -  중앙매출 1위.
   -  객단가 중간.
   -  매출건수 적음.
   - 한번 소비 규모 큼/ 상권당 매출 규모 큼/ 프리미엄 소비 특징
=> 고단가 소비중심 구조

- 2. 발달상권 << 서울 상권 매출의 양적 중심 축 >>
  - 매출건수 압도적 많음.
  - 평균매출은 관광특구보다 낮음.
  - 중앙매출 낮음.
  - 객단가 중상위
=> 서울내 매출건수 기준 핵심소비 거점으로, 방문기반 소비구조가 강하게 나타남.

- 3. 전통시장 << 객단가 대비 낮은 매출규모 구조 >>
  - 매출건수 중간.
  - 평균매출 낮음.
  - 중앙매출 낮음.
  - 객단가  생각보다 높음.
=> 객단가 수준은 유지되나 상권 전체 매출 규모는 상대적으로 낮아 생활밀착형 소비구조가 나타남.

- 4. 골목상권 << 소규모지만 고단가 생활형 소비구조 >>
  - 매출건수 중간.
  - 평균매출 낮음.
  - 중앙매출 낮음.
  - 객단가 1위.
=> 상대적 높은 객단가를 보이나 전체매출 규모는 낮아 지역기반 생활형 소비 특성이 나타난다.

++ 평균 과 중앙값 비교.  인사이트
- 1. 관광특구
  - 평균 = 중앙값
  => 특정 초대형 상권이 아니라 전체적으로 고르게 높은 매출 구조

- 2. 평균 > 중앙값
  => 일부 초대형 상권이 평균을 끌어올림.

==> 발달상권 = 매출 양극화 존재 / 관광특구 = 고른 고매출 구조

++ 객단가
골목 > 전통 > 발달 > 관광

예상 외로 관광이 객단가가 높지 않았음.
추측:
   관광특구는 방문수 기반 소비이고, 골목상권은 생활밀착 고단가소비
   (관광 = 많이 와서 많이 씀, 골목= 적에와도 크게씀 )




In [ ]:
매출_영역.groupby('상권_구분_코드_명')['서비스_업종_코드'].count()

---
* 자치구별 여성, 나이별 
- 프리미엄 (서초, 강남, 용산) 고단가 소비층 
- 타겟대형 (상권중구, 송파) 매출건수+평균매출 상위
- 생활밀착 (노원, 도봉, 은평) 저단가 고빈도 하위권
- 이상치 (노원) 중앙값 특이구조 (생활밀착과 겹침)

In [ ]:
# 성별 비중
성별컬럼 = ['남성_매출_금액', '여성_매출_금액']
성별라벨 = ['남성', '여성']

gender_data = 최신분기.groupby('자치구_코드_명')[성별컬럼].sum()
gender_data.columns = 성별라벨
gender_pct = gender_data.div(gender_data.sum(axis=1), axis=0) * 100

선택구 = ['강남구','서초구','노원구','중구','송파구']

print(gender_pct.loc[선택구].round(1))

In [ ]:
# 연령 비중
선택구 = ['서초구','강남구','용산구','중구','송파구','노원구','도봉구','은평구']
# 연령대 매출 비중
연령컬럼 = ['연령대_10_매출_금액','연령대_20_매출_금액','연령대_30_매출_금액',
        '연령대_40_매출_금액','연령대_50_매출_금액','연령대_60_이상_매출_금액']
연령라벨 = ['10대','20대','30대','40대','50대','60대+']

age_data = 최신분기.groupby('자치구_코드_명')[연령컬럼].sum()
age_data.columns = 연령라벨
age_pct = age_data.div(age_data.sum(axis=1), axis=0) * 100
age_pct = age_pct.loc[선택구]
print(age_pct.loc[선택구].round(1))

In [ ]:
# 성별 매출 비중
성별컬럼 = ['남성_매출_금액', '여성_매출_금액']
성별라벨 = ['남성', '여성']

선택구 = ['서초구','강남구','용산구','중구','송파구','노원구','도봉구','은평구']

gender_data = 최신분기.groupby('자치구_코드_명')[성별컬럼].sum()
gender_data.columns = 성별라벨
gender_pct = gender_data.div(gender_data.sum(axis=1), axis=0) * 100
gender_pct = gender_pct.loc[선택구]

# 연령대 매출 비중
연령컬럼 = ['연령대_10_매출_금액','연령대_20_매출_금액','연령대_30_매출_금액',
        '연령대_40_매출_금액','연령대_50_매출_금액','연령대_60_이상_매출_금액']
연령라벨 = ['10대','20대','30대','40대','50대','60대+']

age_data = 최신분기.groupby('자치구_코드_명')[연령컬럼].sum()
age_data.columns = 연령라벨
age_pct = age_data.div(age_data.sum(axis=1), axis=0) * 100
age_pct = age_pct.loc[선택구]

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# 성별
gender_pct.plot(kind='bar', stacked=True, ax=axes[0],
                color=['steelblue','coral'], width=0.7, edgecolor='white')
axes[0].set_title('자치구별 성별 매출 비중 (%)', fontweight='bold')
axes[0].set_ylabel('%')
axes[0].set_xlabel('')
axes[0].legend(title='성별', bbox_to_anchor=(1.02, 1), loc='upper left')
axes[0].tick_params(axis='x', rotation=0)

# 연령대
age_pct.plot(kind='bar', stacked=True, ax=axes[1],
             colormap='Set2', width=0.7, edgecolor='white')
axes[1].set_title('자치구별 연령대 매출 비중 (%)', fontweight='bold')
axes[1].set_ylabel('%')
axes[1].set_xlabel('')
axes[1].legend(title='연령대', bbox_to_anchor=(1.02, 1), loc='upper left')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
연령컬럼 = ['연령대_10_매출_금액','연령대_20_매출_금액','연령대_30_매출_금액',
        '연령대_40_매출_금액','연령대_50_매출_금액','연령대_60_이상_매출_금액']
연령라벨 = ['10대','20대','30대','40대','50대','60대+']
age_data = 최신분기.groupby('자치구_코드_명')[연령컬럼].sum()
age_data.columns = 연령라벨
age_pct = age_data.div(age_data.sum(axis=1), axis=0) * 100

선택구 = ['강남구','서초구','노원구','중구','송파구']

fig, ax = plt.subplots(figsize=(11, 6))
age_pct.loc[선택구].T.plot(kind='bar', stacked=False, ax=ax,
                           colormap='Set2', width=0.7, edgecolor='white')
ax.set_title('자치구별 연령대 비중 (%)', fontweight='bold')
ax.set_ylabel('%')
ax.set_xlabel('')
ax.legend(title='자치구', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:

연령컬럼 = ['연령대_10_매출_금액','연령대_20_매출_금액','연령대_30_매출_금액',
        '연령대_40_매출_금액','연령대_50_매출_금액','연령대_60_이상_매출_금액']
연령라벨 = ['10대','20대','30대','40대','50대','60대+']

자치구_연령매출 = (
    최신분기
    .groupby('자치구_코드_명')[연령컬럼]
    .sum()
)

자치구_연령비중 = 자치구_연령매출.div(
    자치구_연령매출.sum(axis=1),
    axis=0
)



import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12,8))

sns.heatmap(
    자치구_연령비중,
    cmap='YlOrRd',
    linewidths=0.5
)

plt.title('자치구별 연령대 매출 비중')
plt.ylabel('자치구')
plt.xlabel('연령대')

plt.show()

In [ ]:
# 연령대별 자치구 매출 탑 5
연령컬럼 = ['연령대_10_매출_금액','연령대_20_매출_금액','연령대_30_매출_금액',
        '연령대_40_매출_금액','연령대_50_매출_금액','연령대_60_이상_매출_금액']
연령라벨 = ['10대','20대','30대','40대','50대','60대+']

for 연령, 컬럼 in zip(연령라벨, 연령컬럼):
    상위구 = 최신분기.groupby('자치구_코드_명')[컬럼].sum().nlargest(5).index.tolist()
    print(f'{연령}: {상위구}')

총 인사이트:
유형 A️ 고소득 실무·중산층 중심 (30~40대 중심형)
대표:강남구 서초구 송파구 용산구
강남구 - 30대 23.2 ,  30대 23.2
서초구 - 30대 22.1, 40대 24.9 ,  50대 22.3
송파구 - 30대 22.0, 40대 25.4 ,  60대 21.9
==>경제활동 핵심 연령층 중심 소비 구조
= 고단가 소비 가능 지역
유형 B️ 초고령 소비 중심 생활형 상권
대표:도봉구 서초구 은평구 노원구
도봉구 - 30대 22.0, 40대 25.4 ,  60대 21.9
은평구 - 60대+ 32.4
노원구 - 50대 27.0,  60대 23.8
==>고령층 중심 소비 구조 = 생활밀착형 상권  = 방문형 소비보다 거주형 소비 (지역 기반 소비형 상권)
유형 C️ 관광·업무 중심 혼합형 소비 구조
대표:중구
중구 - 60대+ 29.7 ← 최고, 30대 19.7, 40대 18.9
청년 중심 ❌
중산층 중심 ❌
고령 중심 단순 구조도 ❌
관광 + 업무 + 전통 상권 혼합 소비 구조 (방문형 소비 중심 상권)
유형 D️ 균형형 직주혼합 소비 구조
대표:용산구
용산구 - 20대 13.5
30대 23.5
40대 23.3
50대 22.3
=>(연령 편중 없음
= 유입형 + 거주형 혼합 소비 구조) (서울에서 가장 균형 잡힌 소비 구조 중 하나)

20대 소비 TOP 
청년 소비 중심 지역 = 마포 + 강남축
40대 소비 TOP
경제활동 핵심 소비 중심 = 강남 3구
60대 소비 TOP
중구 반복 등장
전 연령층 소비가 모두 유입되는 지역

총: 자치구별 연령대 매출 비중 분석 결과 서울 상권은 소비 연령 구조에 따라 뚜렷한 유형 차이를 보였다.
강남·서초·송파는 30~40대 중심의 경제활동 핵심 소비 구조가 나타났으며,
마포는 20대 소비 비중이 높은 청년 중심 상권으로 확인되었다.
반면 도봉·은평·노원은 50~60대 비중이 높은 생활밀착형 소비 구조를 보였고,
중구는 전 연령대 소비가 동시에 유입되는 관광·업무 혼합형 방문 소비 중심 상권으로 나타났다.

* 전체연도

In [ ]:
매출_영역.head(2)

In [ ]:
매출_영역['연도'] = 매출_영역['기준_년분기_코드'].astype(str).str[:4].astype(int)

In [ ]:
# 전체 연도 분기로 필터 후 상권 단위 집계

전체_상권집계 = 매출_영역.groupby(
    ['연도','상권_코드','상권_코드_명','상권_구분_코드_명','자치구_코드_명']
).agg(
    총매출=('당월_매출_금액','sum'),
    총건수=('당월_매출_건수','sum'),
    업종수=('서비스_업종_코드','nunique'),
).reset_index()

전체_상권집계['객단가'] = 전체_상권집계['총매출'] / 전체_상권집계['총건수']
전체_상권집계['log_매출'] = np.log10(전체_상권집계['총매출'].clip(lower=1))
전체_상권집계['총매출_억'] = (전체_상권집계['총매출']/1e8).round(1)

print(f'분석 대상 상권: {len(전체_상권집계):,}개')
전체_상권집계.head(5)

In [ ]:
매출_영역['연도'].unique()

In [ ]:
전체_상권집계['연도'].unique()

In [ ]:

import matplotlib.pyplot as plt
import platform

# 한글 폰트
if platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 자치구별 집계
구분통계 = 상권집계.groupby('자치구_코드_명').agg(
    매출건수=('총건수', 'sum'),
    평균매출=('총매출', 'mean'),
    중앙매출=('총매출', 'median'),
    평균객단가=('객단가', 'mean')
).reset_index()

구분통계['평균객단가'] = 구분통계['평균객단가'].round(0)

# 평균매출 기준 정렬
구분통계 = 구분통계.sort_values('평균매출', ascending=True)

fig, axes = plt.subplots(1, 4, figsize=(22, 10))

# 1. 매출건수
axes[0].barh(구분통계['자치구_코드_명'], 구분통계['매출건수'], color='steelblue')
axes[0].set_title('자치구별 매출건수')
axes[0].set_xlabel('건수')

# 2. 평균매출
axes[1].barh(구분통계['자치구_코드_명'], 구분통계['평균매출'] / 1e8, color='coral')
axes[1].set_title('자치구별 평균매출 (억원)')
axes[1].set_xlabel('억원')

# 3. 중앙매출
axes[2].barh(구분통계['자치구_코드_명'], 구분통계['중앙매출'] / 1e8, color='mediumpurple')
axes[2].set_title('자치구별 중앙매출 (억원)')
axes[2].set_xlabel('억원')

# 4. 평균객단가
axes[3].barh(구분통계['자치구_코드_명'], 구분통계['평균객단가'], color='seagreen')
axes[3].set_title('자치구별 평균객단가 (원)')
axes[3].set_xlabel('원')

plt.tight_layout()
plt.show()

In [ ]:
매출_영역.head(2)

* 자치구별 코로나 이전 이후 매출 성장률(전체연도)

In [ ]:
선택구 = ['서초구','강남구','용산구','중구','송파구','노원구','도봉구','은평구']
순서 = 선택구

period = (
    매출_영역[
        (매출_영역['자치구_코드_명'].isin(선택구)) &
        (매출_영역['기준_년분기_코드'].isin([20201, 20244]))
    ]
    .groupby(['자치구_코드_명', '기준_년분기_코드'])['당월_매출_금액']
    .sum()
    .unstack()
)

period.columns = ['2020Q1', '2024Q4']
period['성장률(%)'] = ((period['2024Q4'] / period['2020Q1'] - 1) * 100).round(1)
period['2020Q1_조'] = (period['2020Q1'] / 1e12).round(2)
period['2024Q4_조'] = (period['2024Q4'] / 1e12).round(2)

period = period.reindex(순서)

period[['2020Q1_조', '2024Q4_조', '성장률(%)']]

In [ ]:
import matplotlib.pyplot as plt

plot_df = period[['성장률(%)']].sort_values('성장률(%)', ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(plot_df.index, plot_df['성장률(%)'])
plt.axvline(0, linestyle='--')
plt.title('자치구별 코로나 이후 매출 성장률')
plt.xlabel('성장률(%)')
plt.ylabel('자치구')
plt.show()

* 전체 구조 변화 요약

2020년 1분기 대비 2024년 4분기까지 선택된 8개 자치구 중 대부분 지역에서 상권 규모가 증가하였다. 특히 송파구(63.6%)와 중구(60.2%)는 가장 높은 성장률을 기록하며 빠른 상권 확대가 나타났다. 반면 용산구는 선택된 자치구 중 유일하게 상권 규모가 감소한 지역으로 확인된다.
* 가장 빠르게 성장한 지역

송파구는 2020년 1분기 1.34조에서 2024년 4분기 2.19조로 증가하여 63.6%의 성장률을 기록하였다. 중구 역시 1.24조에서 1.98조로 증가하여 60.2%의 높은 성장률을 보였다. 두 지역 모두 절대 규모 증가와 상대 성장률이 동시에 크게 나타난 지역이다.
* 최대 상권 규모 유지 지역

강남구는 2020년 1분기 2.34조에서 2024년 4분기 2.87조로 증가하여 22.5%의 성장률을 기록하였다. 성장률 자체는 다른 일부 지역보다 낮지만 2024년 기준 가장 큰 상권 규모를 유지하고 있다.
* 강남권 내부 성장 속도 차이

서초구는 1.12조에서 1.55조로 증가하여 38.0%의 성장률을 기록하였다. 이는 같은 강남권에 속하는 강남구(22.5%)보다 높은 성장률로, 동일 권역 내에서도 상권 확대 속도의 차이가 존재함을 보여준다.
* 유일한 감소 지역

용산구는 2020년 1분기 1.49조에서 2024년 4분기 1.24조로 감소하여 –16.4%의 성장률을 기록하였다. 이는 선택된 8개 자치구 중 유일하게 상권 규모가 감소한 사례이다.
* 외곽 주거 중심 지역의 성장 확인

노원구(12.6%), 도봉구(41.4%), 은평구(33.1%) 역시 모두 상권 규모 증가가 나타났다. 특히 도봉구는 비교적 높은 성장률을 기록하여 외곽 주거 중심 지역에서도 상권 확대가 발생했음을 확인할 수 있다.
* 초기 규모와 성장률 간 관계

초기 상권 규모가 큰 지역이 항상 높은 성장률을 보인 것은 아니었다. 예를 들어 강남구는 가장 큰 규모를 유지하고 있지만 성장률은 22.5% 수준이며, 송파구와 중구는 상대적으로 더 높은 성장률을 기록함. 이는 상권 초기 규모와 성장률 사이에 단순한 비례 관계가 나타나지 않음을 보여줌.

* 선택구 간 상권 성장 패턴 차이

선택된 자치구 간 상권 규모 변화는 동일한 방향으로 나타나지 않았다. 일부 지역은 빠른 성장(송파구, 중구), 일부 지역은 안정적 성장(강남구, 서초구), 일부 지역은 감소(용산구) 등 서로 다른 변화 양상이 확인된다. 이는 동일 기간 동안에도 지역별 상권 성장 흐름에 차이가 존재함을 보여준다.


In [ ]:
선택구 = ['서초구','강남구','용산구','중구','송파구','노원구','도봉구','은평구']
매출_영역['연도'] = 매출_영역['기준_년분기_코드']//10

연도별_자치구 = (
    매출_영역[매출_영역['자치구_코드_명'].isin(선택구)]
    .groupby(['연도', '자치구_코드_명'])['당월_매출_금액']
    .sum()
    .unstack() / 1e12
)

연도별_자치구 = 연도별_자치구[선택구]
연도별_자치구.round(2)

In [ ]:
#기본 추이
import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))

연도별_자치구.plot(marker='o')

plt.title('대표 자치구 연도별 총매출 추이 (조 원)')
plt.xlabel('연도')
plt.ylabel('총매출 (조 원)')
plt.xticks(rotation=0)

plt.legend(title='자치구', bbox_to_anchor=(1.02,1), loc='upper left')

plt.grid(True, linestyle='--', alpha=0.4)

plt.show()

In [ ]:
#회복 속도 비교용
연도별_indexed = 연도별_자치구 / 연도별_자치구.loc[2020] * 100

plt.figure(figsize=(12,6))

연도별_indexed.plot(marker='o')

plt.title('대표 자치구 연도별 매출 변화율 (2020=100)')
plt.xlabel('연도')
plt.ylabel('매출 지수 (2019=100)')
plt.xticks(rotation=0)

plt.axhline(100, linestyle='--')

plt.legend(title='자치구', bbox_to_anchor=(1.02,1), loc='upper left')

plt.grid(True, linestyle='--', alpha=0.4)

plt.show()

그룹 A️ 안정적 성장형 (강남·서초·송파)
특징:
2020 → 2023까지 꾸준한 상승
이후 완만한 정체 또는 안정화
예:
강남: 9.90 → 11.68 → 11.14
서초: 4.52 → 6.12 → 6.06
송파: 5.69 → 7.71 → 8.54
=>코로나 이후 빠르게 회복하고 고점 유지하는 “핵심 소비 중심 상권” (서울의 프리미엄 소비축)

그룹 B️ 반등형 (중구)
특징:
가장 강한 회복 흐름
꾸준한 상승 추세 지속
다른 구는 대부분 2023 이후 정체인데
중구는 계속 상승 중
예:
5.18 → 6.04 → 6.54 → 7.23 → 7.85
=>관광·업무 중심 방문형 소비가 코로나 이후 가장 강하게 회복된 지역(서울에서 코로나 영향 가장 컸지만 회복도 가장 강했던 상권)

그룹 C️ 생활형 안정 유지형 (노원·도봉·은평)
특징:
변화폭 작음
외부 유입보다 지역 거주 기반 소비 중심 상권
예:
노원: 1.69 → 1.87 → 1.82
도봉: 0.81 → 1.06 → 1.06
은평: 1.17 → 1.52 → 1.43
=>생활밀착형 상권은 코로나 영향도 작고 회복 변동도 작음

+ 용산구는 "유일한 감소형"
6.13 → 5.98 → 5.47 → 5.01 → 4.72 → 4.65
계속감소중
=>문형 소비 중심 구조 변화 또는 상권 구조 재편 진행 가능성(서울 주요 자치구 중 유일한 구조적 감소 패턴)
+송파구 "조용하지만 강한 성장형"
특징:
2022 잠깐 하락
이후 최고 수준 성장
해석:
거주형 + 대형 상업시설 결합 구조 (강남 다음으로 안정적인 성장형 소비 중심 지역)
+강남 "이미 큰시장인데 더 커진지역"
이미 서울 최대시장인데 또 성장
=> 서울소비 중심축 구조적 강화 (코로나이후 소비집중현상)
요약:
자치구별 연도별 매출 추이를 비교한 결과 서울 상권은 회복 경로에 따라 세 가지 유형으로 구분되었다.
강남·서초·송파는 코로나 이후 빠르게 회복한 안정적 성장형 소비 구조를 보였으며,
중구는 관광·업무 중심 상권 특성에 따라 가장 강한 반등 흐름이 나타났다.
반면 노원·도봉·은평 등 생활밀착형 상권은 변화폭이 작고 안정적인 소비 구조를 유지하였다.
특히 용산구는 주요 자치구 중 유일하게 지속적인 감소 흐름을 보여 상권 구조 변화 가능성이 확인되었다.

---

# 상권변화 지표

In [ ]:
from pathlib import Path

current = Path.cwd()
project_root = current if (current / 'data').exists() else current.parent

csv_path = project_root / 'data' / '08_상권변화지표' / '서울시 상권분석서비스(상권변화지표-상권).csv'

df = pd.read_csv(csv_path, encoding='cp949')
print(f'✅ 로드 완료: {df.shape}')

In [ ]:
area

In [ ]:
# 영역 정보 merge
매출_영역 = sales.merge(
    area[['상권_코드','자치구_코드_명','행정동_코드_명','엑스좌표_값','와이좌표_값']],
    on='상권_코드', how='left'
)
print(f'merge 후: {매출_영역.shape}')
print(f'결측 자치구: {매출_영역["자치구_코드_명"].isna().sum()}')

In [ ]:
df.columns

In [ ]:
print(area.columns.tolist())

In [ ]:
상권_통합 = area.merge(
    df[['기준_년분기_코드','상권_코드','상권_변화_지표_명','운영_영업_개월_평균','폐업_영업_개월_평균','서울_운영_영업_개월_평균','서울_폐업_영업_개월_평균']],
    on='상권_코드',
    how='left'
)
상권_통합.head(2)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))

선택구 = ['서초구','강남구','용산구','중구','송파구','노원구','도봉구','은평구']

# 선택 자치구만 필터링
filtered = 상권_통합[
    상권_통합['자치구_코드_명'].isin(선택구)
]

# 자치구별 변화지표 개수 계산
pivot = (
    filtered
    .groupby(['자치구_코드_명', '상권_변화_지표_명'])
    .size()
    .unstack()
)

# 막대 그래프
pivot.plot(kind='bar', ax=ax)

ax.set_title('자치구별 변화지표 분포')
ax.set_xlabel('자치구')
ax.set_ylabel('상권 개수')
ax.legend(title='변화지표')

plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
pivot_ratio = pivot.div(pivot.sum(axis=1), axis=0)

pivot_ratio.plot(kind='bar', stacked=True, figsize=(10,6))

plt.title('자치구별 변화지표 비율 분포')
plt.xlabel('자치구')
plt.ylabel('비율')
plt.legend(title='변화지표', bbox_to_anchor=(1.02,1))
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

전체 구조 요약

선택된 8개 자치구 모두에서 상권 변화 유형의 분포 패턴이 서로 다르게 나타난다.
특히 일부 자치구는 특정 변화 유형이 다른 유형보다 매우 크게 나타나 상권 변화 구조의 차이가 존재함을 확인할 수 있다.

강남구: 다이나믹 유형 최다
강남구에서는 다이나믹 유형이 가장 많이 나타난 변화 유형이다.
또한 정체·상권축소·상권확장보다 다이나믹 유형의 빈도가 가장 크게 나타난다.

노원구: 다이나믹 중심 구조
노원구에서도 다이나믹 유형이 가장 많이 나타난 변화 유형이다.
정체 유형은 상대적으로 가장 적게 나타난다.

도봉구: 상권축소와 다이나믹이 동시에 높은 구조
도봉구에서는 다이나믹 유형과 상권축소 유형의 빈도가 유사하게 높게 나타난다.
상권확장 유형은 가장 적은 비중을 차지한다.

서초구: 정체 유형 최다
서초구에서는 정체 유형이 가장 많이 나타난 변화 유형이다.
다른 세 유형보다 정체 유형이 가장 높은 빈도를 보인다.

송파구: 다이나믹 중심 구조
송파구에서는 다이나믹 유형이 가장 많이 나타난 변화 유형이다.
상권확장 유형은 상대적으로 가장 적게 나타난다.

용산구: 정체와 상권확장이 함께 높은 구조
용산구에서는 정체 유형이 가장 많이 나타난 변화 유형이다.
상권확장과 다이나믹 유형의 빈도는 유사한 수준으로 나타난다.

은평구: 다이나믹과 정체 중심 구조
은평구에서는 다이나믹 유형이 가장 많이 나타난 변화 유형이며, 정체 유형도 비교적 높은 빈도로 나타난다.

중구: 정체 유형 압도적 우세
중구에서는 정체 유형이 다른 모든 유형보다 매우 크게 나타난다.
특히 다이나믹과 상권축소 유형은 상대적으로 낮은 빈도로 나타난다.

In [ ]:
# 연도 생성
상권_통합['연도'] = 상권_통합['기준_년분기_코드'] // 10

# 선택 자치구
선택구 = ['서초구','강남구','용산구','중구','송파구','노원구','도봉구','은평구']

상권_통합_filtered = 상권_통합[상권_통합['자치구_코드_명'].isin(선택구)]

# 전체 상권 개수
total = 상권_통합_filtered.groupby(['연도','자치구_코드_명']).size()

# 축소 상권 개수
축소 = 상권_통합_filtered[상권_통합_filtered['상권_변화_지표_명'] == '상권축소'] \
        .groupby(['연도','자치구_코드_명']).size()

# 비율 계산
축소_비율 = (축소 / total * 100).unstack()

축소_비율_T = 축소_비율.T.round(2)
print(축소_비율_T)

In [ ]:
import matplotlib.pyplot as plt

# 연도 생성
상권_통합['연도'] = 상권_통합['기준_년분기_코드'] // 10

# 선택 자치구
선택구 = ['서초구','강남구','용산구','중구','송파구','노원구','도봉구','은평구']

상권_통합_filtered = 상권_통합[상권_통합['자치구_코드_명'].isin(선택구)]

# 전체 상권 개수
total = 상권_통합_filtered.groupby(['연도','자치구_코드_명']).size()

# 축소 상권 개수
축소 = 상권_통합_filtered[상권_통합_filtered['상권_변화_지표_명'] == '상권축소'] \
        .groupby(['연도','자치구_코드_명']).size()

# 비율 계산
축소_비율 = (축소 / total * 100).unstack()

# 그래프
plt.figure(figsize=(12,6))

축소_비율.plot(marker='o')

plt.axvline(x=2020, linestyle='--', color='gray', label='코로나 시작')

plt.title('자치구별 상권축소 비율 추이')
plt.xlabel('연도')
plt.ylabel('상권축소 비율 (%)')

plt.legend(title='자치구', bbox_to_anchor=(1.02,1))
plt.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

상권축소 비율이 가장 높은 지역: 도봉구
도봉구는 모든 연도에서 다른 자치구보다 높은 수준의 상권축소 비율이 나타난 지역

상권축소 비율이 가장 낮은 지역: 중구
중구는 선택된 자치구 중 가장 낮은 수준의 상권축소 비율이 지속적으로 나타난 지역

=>선택된 8개 자치구 모두에서 2019년 대비 2025년 기준 상권축소 비율이 감소함

감소폭 큰 지역
은평, 노원, 서초, 송파, 강남
-> 은평 과 노원은 다른 자치구보다 상대적 큰 감소폭이 나타난 지역

기간 중 가장 높은 상권축소 비율이 나타난 시점
도봉구 : 20년 22년
송파구 : 21년
노원구 : 20년
서초구 : 21년
강남구 : 23년
->일부 자치구에서는 2020년 또는 2021년 시기에 상대적으로 높은 상권축소 비율이 나타남.

전체기간동안 낮은 수준의 유지 지역: 중구, 용산

지역간 구조적 차이 
도봉구 약 30%  vs 중구 약 2~7%

요약:
2019년부터 2025년까지 자치구별 상권축소 비율을 비교한 결과, 모든 선택 자치구에서 상권축소 비율이 전반적으로 감소하는 흐름이 나타났다. 특히 은평구와 노원구는 감소폭이 크게 나타났으며, 도봉구는 전체 기간 동안 가장 높은 상권축소 비율을 유지한 반면 중구와 용산구는 지속적으로 낮은 수준을 유지하였다. 

In [ ]:
# 연도 생성
상권_통합['연도'] = 상권_통합['기준_년분기_코드'] // 10

# 선택 자치구
선택구 = ['서초구','강남구','용산구','중구','송파구','노원구','도봉구','은평구']

상권_통합_filtered = 상권_통합[상권_통합['자치구_코드_명'].isin(선택구)]

# 전체 상권 개수
total = 상권_통합_filtered.groupby(['연도','자치구_코드_명']).size()

# 확장 상권 개수
확장 = 상권_통합_filtered[상권_통합_filtered['상권_변화_지표_명'] == '상권확장'] \
        .groupby(['연도','자치구_코드_명']).size()

# 비율 계산
확장_비율 = (확장 / total * 100).unstack()

확장_비율_T = 확장_비율.T.round(2)
print(확장_비율_T)

In [ ]:
import matplotlib.pyplot as plt

# 연도 생성
상권_통합['연도'] = 상권_통합['기준_년분기_코드'] // 10

# 선택 자치구
선택구 = ['서초구','강남구','용산구','중구','송파구','노원구','도봉구','은평구']

상권_통합_filtered = 상권_통합[상권_통합['자치구_코드_명'].isin(선택구)]

# 전체 상권 개수
total = 상권_통합_filtered.groupby(['연도','자치구_코드_명']).size()

# 확장 상권 개수
확장 = 상권_통합_filtered[상권_통합_filtered['상권_변화_지표_명'] == '상권확장'] \
        .groupby(['연도','자치구_코드_명']).size()

# 비율 계산
확장_비율 = (확장 / total * 100).unstack()

# 그래프
plt.figure(figsize=(12,6))

확장_비율.plot(marker='o')

plt.axvline(x=2020, linestyle='--', color='gray', label='코로나 시작')

plt.title('자치구별 상권확장 비율 추이')
plt.xlabel('연도')
plt.ylabel('상권확장 비율 (%)')

plt.legend(title='자치구', bbox_to_anchor=(1.02,1))
plt.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

상권확장 비율 인사이트:
많은 구들이 21~23구간에서 약해졌다가 24~25구간에서 회복조짐을 보임.
=>확장비율을 보면 대부분 자치구에서 2021년부터 2023년 사이 감소 흐름이 나타났고, 2024년 이후 일부 지역에서는 다시 회복되는 움직임이 보임.

용산구 - 높은 확장비율권. 장기적으로 확장성이 강한 지역.
서초구 - 높은 확장비율 유지, 최근 상승흐름. 회복된 안정적인 지역.
도봉구+ 노원구 - 최근 확장비율 폭이 가장 큼. 확장움직임 상대적으로 활발한 지역.
송파구 - 확장비율 감소폭이 가장큼. 회복도 제한적. 회복 불가중.
중구 + 은평구 - 지속적인 확장비율 감소중.

In [ ]:
# 선택 자치구
선택구 = ['서초구','강남구','용산구','중구','송파구','노원구','도봉구','은평구']

# 선택구만 필터링
상권_통합_filtered = 상권_통합[상권_통합['자치구_코드_명'].isin(선택구)]

# 자치구별 평균 계산
평균 = 상권_통합_filtered.groupby('자치구_코드_명')[
    ['운영_영업_개월_평균', '폐업_영업_개월_평균']
].mean()

# 순서 맞추기 (보기 좋게)
평균 = 평균.loc[선택구]
평균

In [ ]:
import matplotlib.pyplot as plt

# 선택 자치구
선택구 = ['서초구','강남구','용산구','중구','송파구','노원구','도봉구','은평구']

# 선택구만 필터링
상권_통합_filtered = 상권_통합[상권_통합['자치구_코드_명'].isin(선택구)]

# 자치구별 평균 계산
평균 = 상권_통합_filtered.groupby('자치구_코드_명')[
    ['운영_영업_개월_평균', '폐업_영업_개월_평균']
].mean()

# 순서 맞추기 (보기 좋게)
평균 = 평균.loc[선택구]

# 그래프
fig, ax = plt.subplots(figsize=(10, 6))

평균.plot(kind='bar', ax=ax)

ax.set_title('대표 자치구별 운영/폐업 영업개월 평균 비교')
ax.set_xlabel('자치구')
ax.set_ylabel('개월 수')

ax.legend(['운영 평균', '폐업 평균'])

plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

자치구별 평ㄱ균 영업개월 인사이트
운영 점포 평균 영업기간 전체적으로 8-10년 수준
폐업 점포 평균 영업기간 전체적 4-5년 수준
-> 단기 폐업보단 일정기간 운영후 폐업구조

+장기운영 점포 비중 높음 -> 용산 , 서초
+ 운영 폐업 모두 낮음. -> 강납
+모든 자치구에서 운영 점포가 폐업 점포보다 더 오래 유지됨

자치구별 운영 영업개월 평균과 폐업 영업개월 평균을 비교한 결과 자치구별 상권 유지 구조에서 차이가 나타났다.

중구는 운영 영업개월 평균과 폐업 영업개월 평균이 모두 가장 높게 나타나 다른 자치구 대비 점포 유지 기간이 가장 긴 상권 구조를 보였다. 이는 점포 교체 속도가 상대적으로 낮은 안정적인 상권 구조로 해석된다.

서초구 역시 운영 영업개월 평균이 비교적 높은 수준으로 나타나 점포 유지 기간이 긴 안정형 상권 특성을 보였다.

반면 강남구는 운영 영업개월 평균과 폐업 영업개월 평균이 상대적으로 낮게 나타나 다른 자치구 대비 점포 교체 속도가 빠른 구조로 나타났다.

송파구는 운영 영업개월 평균과 폐업 영업개월 평균이 중간 수준으로 나타나 안정성과 변화가 함께 나타나는 상권 구조로 해석된다.

노원구, 도봉구, 은평구는 운영 영업개월 평균과 폐업 영업개월 평균이 서로 유사한 수준으로 나타났으며 전반적으로 큰 변동 없이 안정적인 상권 유지 구조를 보였다.

##  직장인구/상주인구

In [ ]:
from pathlib import Path
import zipfile
import shutil

# 프로젝트 루트 감지
current = Path.cwd()
project_root = current if (current / 'data').exists() else current.parent

# 경로 설정
target_dir = project_root / 'data' / '06_직장인구'
csv_path = target_dir / '서울시 상권분석서비스(직장인구-상권).csv'

# CSV 없으면 raw/ 에서 zip 찾아서 압축 해제
if not csv_path.exists():
    print('📦 CSV 없음, zip 파일 찾는 중...')
    raw_dir = project_root / 'data' / 'raw'
    
    # 직장인구 관련 zip 찾기
    zip_files = list(raw_dir.glob('*직장인구*.zip'))
    if not zip_files:
        raise FileNotFoundError(f'{raw_dir} 에 직장인구 zip 없음')
    
    zip_path = zip_files[0]
    print(f'  압축 해제: {zip_path.name}')
    target_dir.mkdir(parents=True, exist_ok=True)
    
    # Mac zip의 한글 파일명 문제 해결
    with zipfile.ZipFile(zip_path, 'r') as z:
        for info in z.infolist():
            if info.filename.startswith('__MACOSX'):
                continue
            try:
                filename = info.filename.encode('cp437').decode('utf-8')
            except:
                filename = info.filename
            
            if info.is_dir():
                continue
            
            out_path = target_dir / filename
            out_path.parent.mkdir(parents=True, exist_ok=True)
            with z.open(info) as src, open(out_path, 'wb') as dst:
                shutil.copyfileobj(src, dst)

# CSV 로드
df_직장 = pd.read_csv(csv_path, encoding='cp949')
print(f'✅ 직장인구 로드 완료: {df_직장.shape}')

In [ ]:
상권_통합[['상권_코드','기준_년분기_코드']].dtypes
df_직장[['상권_코드','기준_년분기_코드']].dtypes

In [ ]:
직장_상권_통합 = 상권_통합.merge(
    df_직장,
    on=['상권_코드', '기준_년분기_코드'],
    how='left'
)

In [ ]:
직장_상권_통합.isna().sum()

In [ ]:
missing_rows = 직장_상권_통합[직장_상권_통합['총_직장_인구_수'].isna()]

missing_rows[['기준_년분기_코드', '상권_코드', '상권_코드_명_x', '자치구_코드_명']].head(20)

In [ ]:
missing_rows['기준_년분기_코드'].value_counts().sort_index()

In [ ]:
missing_rows['상권_구분_코드_명_x'].value_counts()

* merge 
-> 1프로미만 : 양호
전통시장 ->직장인인구 데이터에서 자연스러운 구조.
분기별 누락패턴이 "균등함" 모든분기에서 9~15개 수준 -> 특정 상권코드 자체가 직장인구 데이터에 존재하지 않는 구조

In [ ]:
직장_상권_통합.groupby('상권_변화_지표_명')[
    ['연령대_20_직장_인구_수',
     '연령대_30_직장_인구_수',
     '연령대_40_직장_인구_수']
].mean()
직장_상권_통합

In [ ]:


# 상권 유형별 총 직장인구 평균 계산
선택구 = ['서초구','강남구','용산구','중구','송파구','노원구','도봉구','은평구']

평균_직장 = (
    직장_상권_통합[
        직장_상권_통합['자치구_코드_명'].isin(선택구)
    ]
    .groupby('자치구_코드_명')['총_직장_인구_수']
    .mean()
    .reset_index()
)
# 막대 그래프
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(평균_직장['자치구_코드_명'], 평균_직장['총_직장_인구_수'],
       color=['#4C72B0', '#DD8452', '#55A868', '#C44E52'])

ax.set_title('자치구별 평균 직장인구')
ax.set_xlabel('자치구')
ax.set_ylabel('평균 직장인구 수 (명)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

+직장인 자치구별 인구수
상위 - 강남, 중구
중상 - 서초,송파
중 - 용산 
외. - 직장인 인구수 적음

In [ ]:
선택구 = ['서초구','강남구','용산구','중구','송파구','노원구','도봉구','은평구']

연도별 = (
    직장_상권_통합[
        직장_상권_통합['자치구_코드_명'].isin(선택구)
    ]
    .groupby(['연도', '자치구_코드_명'])['총_직장_인구_수']
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10,6))

for 구 in 선택구:
    df_구 = 연도별[연도별['자치구_코드_명'] == 구]
    ax.plot(
        df_구['연도'],
        df_구['총_직장_인구_수'],
        marker='o',
        label=구
    )

ax.axvline(x=2020, linestyle='--', label='코로나 시작')

ax.set_title('선택 자치구별 연도별 직장인구 추이')
ax.set_xlabel('연도')
ax.set_ylabel('평균 직장인구 수 (명)')

ax.legend()
plt.tight_layout()
plt.show()

연도별로보는 자치구별 직작인 인구 추이(연도)
자치구별로 평균 직장인구 수 상위권, 강남+중구 비교 - 강남구에 직장인 인구 상승. 중구 직장인인구 최근 하락
                     중상위, 서초+ 송파 - 서초 유지, 송파 상승.
                      중 , 용산 상승
                     하위 , 노원+동봉+은평 인구수 적고 낮음.

요약:직장인 인구는 강남·중구 등 중심 업무지역에 집중되어 있으며, 송파·용산은 증가 흐름이 나타나는 확장형 지역으로 확인되고, 노원·도봉·은평은 상대적으로 낮은 규모가 유지되는 지역으로 나타난다.

# 상구인구

In [ ]:
from pathlib import Path
import zipfile, shutil

current = Path.cwd()
project_root = current if (current / 'data').exists() else current.parent

# 경로
target_dir = project_root / 'data' / '07_상주인구'
csv_path = target_dir / '서울시 상권분석서비스(상주인구-상권).csv'

# CSV 없으면 raw에서 zip 찾아서 자동 압축 해제
if not csv_path.exists():
    print('📦 CSV 없음, zip 찾는 중...')
    raw_dir = project_root / 'data' / 'raw'
    
    zip_files = list(raw_dir.glob('*상주인구*.zip'))
    if not zip_files:
        raise FileNotFoundError(f'{raw_dir} 에 상주인구 zip 없음')
    
    zip_path = zip_files[0]
    print(f'  압축 해제: {zip_path.name}')
    target_dir.mkdir(parents=True, exist_ok=True)
    
    with zipfile.ZipFile(zip_path, 'r') as z:
        for info in z.infolist():
            if info.filename.startswith('__MACOSX'):
                continue
            try:
                filename = info.filename.encode('cp437').decode('utf-8')
            except:
                filename = info.filename
            
            if info.is_dir():
                continue
            
            out_path = target_dir / filename
            out_path.parent.mkdir(parents=True, exist_ok=True)
            with z.open(info) as src, open(out_path, 'wb') as dst:
                shutil.copyfileobj(src, dst)
    print(f'  ✅ 완료')

# CSV 로드
df_상주 = pd.read_csv(csv_path, encoding='cp949')
print(f'✅ 상주인구 로드: {df_상주.shape}')

In [ ]:
상주_상권_통합 = 상권_통합.merge(
    df_상주[
        [
            '상권_코드', '기준_년분기_코드',
            '총_상주인구_수',
            '남성_상주인구_수', '여성_상주인구_수',
            '연령대_10_상주인구_수', '연령대_20_상주인구_수',
            '연령대_30_상주인구_수', '연령대_40_상주인구_수',
            '연령대_50_상주인구_수', '연령대_60_이상_상주인구_수',
            '남성연령대_10_상주인구_수', '남성연령대_20_상주인구_수',
            '남성연령대_30_상주인구_수', '남성연령대_40_상주인구_수',
            '남성연령대_50_상주인구_수', '남성연령대_60_이상_상주인구_수',
            '여성연령대_10_상주인구_수', '여성연령대_20_상주인구_수',
            '여성연령대_30_상주인구_수', '여성연령대_40_상주인구_수',
            '여성연령대_50_상주인구_수', '여성연령대_60_이상_상주인구_수',
            '총_가구_수', '아파트_가구_수', '비_아파트_가구_수'
        ]
    ],
    on=['상권_코드', '기준_년분기_코드'],
    how='left'
)

In [ ]:
len(상주_상권_통합.columns)

In [ ]:
print(상권_통합.shape)
print(상주_상권_통합.shape)
print(상주_상권_통합['총_상주인구_수'].isna().sum())
print(상주_상권_통합['총_상주인구_수'].isna().mean() * 100)

In [ ]:
상주_상권_통합['총_상주인구_수'].isna().groupby(
    상주_상권_통합['자치구_코드_명']
).sum()

In [ ]:
print(상주_상권_통합.columns.tolist())

In [ ]:
상주_상권_통합['총_상주인구_수'].isna().groupby(
    상주_상권_통합['상권_구분_코드_명']
).sum()

상주인구 데이터 결측은 특정 자치구에 집중되지 않고 서울 전역에 고르게 분포하고 있어 merge 오류가 아닌 데이터 제공 단위 차이에 따른 구조적 결측으로 판단됨.
따라 상주인구는 전체 상권 공통설명 변수라기보다 일부 상권유형에 선택적으로  적용가능한 보조 설명변수로 활용하는게 적절함.

In [ ]:
# 자치구별 총 상주인구 평균 계산
선택구 = ['서초구','강남구','용산구','중구','송파구','노원구','도봉구','은평구']
평균_상주 = (
    상주_상권_통합[
        상주_상권_통합['자치구_코드_명'].isin(선택구)
    ]
    .groupby('자치구_코드_명')['총_상주인구_수']
    .mean()
    .reset_index()
)

# 막대 그래프
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(평균_상주['자치구_코드_명'], 평균_상주['총_상주인구_수'],
       color=['#4C72B0', '#DD8452', '#55A868', '#C44E52'])

ax.set_title('자치구별 평균 상주인구')
ax.set_xlabel('자치구')
ax.set_ylabel('평균 상주인구 수 (명)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

자치별구 평균상주인구수 비교
상위 탑 3 송파 > 은평 > 도봉
중간     서초와 용산 > 강남과 노원 
하위      중구

In [ ]:
선택구 = ['서초구','강남구','용산구','중구','송파구','노원구','도봉구','은평구']

연도별 = (
    상주_상권_통합[
        상주_상권_통합['자치구_코드_명'].isin(선택구)
    ]
    .groupby(['연도', '자치구_코드_명'])['총_상주인구_수']
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10,6))

for 구 in 선택구:
    df_구 = 연도별[연도별['자치구_코드_명'] == 구]
    ax.plot(
        df_구['연도'],
        df_구['총_상주인구_수'],
        marker='o',
        label=구
    )

ax.axvline(x=2020, linestyle='--', label='코로나 시작')

ax.set_title('선택 자치구별 연도별 상주인구 추이')
ax.set_xlabel('연도')
ax.set_ylabel('평균 상주인구 수 (명)')

ax.legend()
plt.tight_layout()
plt.show()

연도별 자치별구 평균상주인구수 비교
상위 탑 3 = 송파 > 은평 > 도봉     ->(추이비교) 송파(살짝 떨어지고 유지), 은평(살짝떨어지고유지) , 도봉 (살짝떨어지고 유지)
중간      = 서초와 용산 > 강남과 노원 ->(추이비교) 거의 유지
하위      =  중구                 ->(추이비교) 유지

# 길단위

In [ ]:
from pathlib import Path

current = Path.cwd()
project_root = current if (current / 'data').exists() else current.parent

csv_path = project_root / 'data' / 'extracted' /'서울시 상권분석서비스(길단위인구-상권)/서울시 상권분석서비스(길단위인구-상권).csv'

df_길단위 = pd.read_csv(csv_path, encoding='cp949')
print(f'✅ 로드 완료: {df.shape}')

In [ ]:
df_길단위.head(2)

In [ ]:
print("평균:", df_길단위['총_유동인구_수'].mean())
print("중앙값:", df_길단위['총_유동인구_수'].median())
print("최대값:", df_길단위['총_유동인구_수'].max())

In [ ]:
import numpy as np
df_길단위['log_유동인구'] = np.log1p(df_길단위['총_유동인구_수'])
df_길단위['log_유동인구']

In [ ]:
유동_상권_통합 = 상권_통합.merge(
    df_길단위[
        [
            '상권_코드',
            '기준_년분기_코드',
            '총_유동인구_수',
            '남성_유동인구_수',
            '여성_유동인구_수',
            '연령대_10_유동인구_수',
            '연령대_20_유동인구_수',
            '연령대_30_유동인구_수',
            '연령대_40_유동인구_수',
            '연령대_50_유동인구_수',
            '연령대_60_이상_유동인구_수',
            '시간대_00_06_유동인구_수',
            '시간대_06_11_유동인구_수',
            '시간대_11_14_유동인구_수',
            '시간대_14_17_유동인구_수',
            '시간대_17_21_유동인구_수',
            '시간대_21_24_유동인구_수',
            '월요일_유동인구_수',
            '화요일_유동인구_수',
            '수요일_유동인구_수',
            '목요일_유동인구_수',
            '금요일_유동인구_수',
            '토요일_유동인구_수',
            '일요일_유동인구_수',
            'log_유동인구'
        ]
    ],
    on=['상권_코드', '기준_년분기_코드'],
    how='left'
)

In [ ]:
print(상권_통합.shape)
print(유동_상권_통합.shape)
print(유동_상권_통합['총_유동인구_수'].isna().mean()*100)

In [ ]:
선택구 = ['서초구','강남구','용산구','중구','송파구','노원구','도봉구','은평구']

연도별 = (
    유동_상권_통합[
        유동_상권_통합['자치구_코드_명'].isin(선택구)
    ]
    .groupby(['연도', '자치구_코드_명'])['총_유동인구_수']
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10,6))

for 구 in 선택구:
    df_구 = 연도별[연도별['자치구_코드_명'] == 구]
    ax.plot(
        df_구['연도'],
        df_구['총_유동인구_수'],
        marker='o',
        label=구
    )

ax.axvline(x=2020, linestyle='--', label='코로나 시작')

ax.set_title('선택 자치구별 연도별 유동인구 추이')
ax.set_xlabel('연도')
ax.set_ylabel('평균 유동인구 수 (명)')

ax.legend()
plt.tight_layout()
plt.show()

전반적인 평균 유동인구수가 떨어지는 추세(21년)로 보인다.

자치구별 유동인구는 송파구가 전 기간 가장 높은 수준을 유지한 반면 도봉구·은평구·노원구는 전반적인 감소 흐름이 나타났고 강남구와 서초구는 비교적 안정적인 수준을 유지하는 특징이 확인

# 상권분석 서비스

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

plt.rcParams['font.family'] = 'AppleGothic'  # Mac 한글 폰트
plt.rcParams['axes.unicode_minus'] = False    # 마이너스 기호 깨짐 방지

In [ ]:
from pathlib import Path
import zipfile
import shutil

# 경로 설정
current = Path.cwd()
project_root = current if (current / 'data').exists() else current.parent
target_dir = project_root / 'data' / 'extracted' / '상권분석서비스(점포_상권)'
zip_path = project_root / 'data' / 'raw' / '상권분석서비스(점포_상권).zip'

# 1. zip 한 번만 풀기 (이미 풀려있으면 스킵)
if not target_dir.exists() or not any(target_dir.iterdir()):
    print(f'📦 압축 해제 중: {zip_path.name}')
    target_dir.mkdir(parents=True, exist_ok=True)
    
    with zipfile.ZipFile(zip_path, 'r') as z:
        for info in z.infolist():
            if info.filename.startswith('__MACOSX'):
                continue
            try:
                filename = info.filename.encode('cp437').decode('utf-8')
            except:
                filename = info.filename
            if info.is_dir():
                continue
            out_path = target_dir / filename
            out_path.parent.mkdir(parents=True, exist_ok=True)
            with z.open(info) as src, open(out_path, 'wb') as dst:
                shutil.copyfileobj(src, dst)
    print('  ✅ 완료')

# 2. 풀린 폴더 안의 모든 CSV 자동 통합
import glob
csv_files = sorted(target_dir.glob('*.csv'))
print(f'\n📂 CSV 파일 {len(csv_files)}개 발견:')
for f in csv_files:
    print(f'  {f.name}')

# 3. 모두 로드해서 통합 (인코딩 자동 시도)
dfs = []
for csv_path in csv_files:
    try:
        df = pd.read_csv(csv_path, encoding='cp949', low_memory=False)
    except UnicodeDecodeError:
        df = pd.read_csv(csv_path, encoding='utf-8', low_memory=False)
    dfs.append(df)
    print(f'✅ {csv_path.name}: {df.shape}')

df_전체 = pd.concat(dfs, ignore_index=True)
print(f'\n📊 전체 데이터: {df_전체.shape}')

In [ ]:
점포_집계 = (
    df_전체
    .groupby(['상권_코드', '기준_년분기_코드'])
    .agg(
        총점포수=('점포_수', 'sum'),
        총유사업종점포수=('유사_업종_점포_수', 'sum'),
        총개업점포수=('개업_점포_수', 'sum'),
        총폐업점포수=('폐업_점포_수', 'sum'),
        총프랜차이즈점포수=('프랜차이즈_점포_수', 'sum')
    )
    .reset_index()
)

점포_집계['개업률'] = (점포_집계['총개업점포수'] / 점포_집계['총점포수'] * 100).round(2)
점포_집계['폐업률'] = (점포_집계['총폐업점포수'] / 점포_집계['총점포수'] * 100).round(2)
점포_집계['순성장률'] = (점포_집계['개업률'] - 점포_집계['폐업률']).round(2)

점포_집계

In [ ]:
상권_통합_2024 = 상권_통합[상권_통합['기준_년분기_코드'] <= 20244].copy()

점포_상권_통합 = 상권_통합_2024.merge(
    점포_집계,
    on=['상권_코드', '기준_년분기_코드'],
    how='left'
)

In [ ]:
print(상권_통합_2024.shape)
print(점포_상권_통합.shape)

print(점포_상권_통합[['총점포수', '총개업점포수', '총폐업점포수', '순성장률']].isna().sum())
print((점포_상권_통합['총점포수'].isna().mean() * 100).round(2))

In [ ]:
# 점포수 많으면 성장률이 높나?
점포_상권_통합[['총점포수','순성장률']].corr()

In [ ]:
# 자치구별 성장률
점포_상권_통합.groupby('자치구_코드_명')['순성장률'].mean()

In [ ]:
# 자치구별 평균 점포수

# 연도 + 자치구별 평균 점포수
result = 점포_상권_통합.groupby(
    ['연도', '자치구_코드_명']
)['총점포수'].mean().reset_index()

# 피벗
pivot_store = result.pivot(
    index='자치구_코드_명',
    columns='연도',
    values='총점포수'
)

print(pivot_store)

In [ ]:
result_sum = 점포_상권_통합.groupby(
    ['연도','자치구_코드_명']
)['총점포수'].sum().reset_index()

result_sum

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))
sns.heatmap(pivot_store, annot=True, fmt=".0f", cmap="Blues")
plt.title("자치구별 평균 점포수 변화")
plt.show()

점포수 변화
상위 중구, 강나,서초 , 송파구
그위는 점포수 낮음.

점포수 상위권 구 중에서 나머지는 다 점포수 오르는 중인데, 21년도 이후로 중구는 낮아지는 추세

In [ ]:
growth = 점포_상권_통합.groupby(
    ['연도','자치구_코드_명']
)['순성장률'].mean().reset_index()

In [ ]:
선택구_list = ['서초구','강남구','용산구','중구','송파구','노원구','도봉구','은평구']

점포_상권_선택구 = 점포_상권_통합[
    점포_상권_통합['자치구_코드_명'].isin(선택구_list)
].copy()

In [ ]:
import matplotlib.pyplot as plt

growth = 점포_상권_선택구.groupby(
    ['연도','자치구_코드_명']
)['순성장률'].mean().reset_index()

plt.figure(figsize=(10,6))

for gu in growth['자치구_코드_명'].unique():
    temp = growth[growth['자치구_코드_명']==gu]
    plt.plot(temp['연도'], temp['순성장률'], marker='o', label=gu)

plt.axvline(x=2020, linestyle='--')

plt.title('선택구별 연도별 순성장률 변화')
plt.xlabel('연도')
plt.ylabel('순성장률')
plt.legend()
plt.show()

24년도 동시하락된 순성장률.
노원구- 20년 가장 최고점 순성장률을 찍고 후 변동성 크게 나타남. (변동성 전체 자치구 중 가장 큰폭)
강남+송파 - 29-23 안정적 유지 하다가 24 하락
도봉구- 19 이후부터 점진적 감소
은평 - 20년붜 성장률 최대 감소.
중구 - 전체흐름+전체구에서 가장 낮은 성장률.

요약:
자치구별 순성장률은 2024년에 대부분 지역에서 동시에 하락하는 흐름이 나타났으며, 노원구는 가장 큰 변동폭을 보였고 도봉구와 송파구는 장기적인 감소 흐름

In [ ]:
open_rate = 점포_상권_선택구.groupby(
    ['연도','자치구_코드_명']
)['개업률'].mean().reset_index()

plt.figure(figsize=(10,6))

for gu in open_rate['자치구_코드_명'].unique():
    temp = open_rate[open_rate['자치구_코드_명']==gu]
    plt.plot(temp['연도'], temp['개업률'], marker='o', label=gu)

plt.axvline(x=2020, linestyle='--')

plt.title('선택구별 연도별 개업률 변화')
plt.xlabel('연도')
plt.ylabel('개업률')
plt.legend()
plt.show()

In [ ]:
close_rate = 점포_상권_선택구.groupby(
    ['연도','자치구_코드_명']
)['폐업률'].mean().reset_index()

plt.figure(figsize=(10,6))

for gu in close_rate['자치구_코드_명'].unique():
    temp = close_rate[close_rate['자치구_코드_명']==gu]
    plt.plot(temp['연도'], temp['폐업률'], marker='o', label=gu)

plt.axvline(x=2020, linestyle='--')

plt.title('선택구별 연도별 폐업률 변화')
plt.xlabel('연도')
plt.ylabel('폐업률')
plt.legend()
plt.show()

자치구별 개업률 폐업률 추이(연도) 인사이트
<개업률> 노원 20년도에 최고점 + 대부분 21~22년도에 개업률 하락 발생 + 23년도 일부개업률 회복 햇다가 24년도 다시 하락하는 추세.
<폐업률> 22년 최저 기록 (강남,서초,용산,노원,중구) 동시저점 발생 (22년도에 폐업률 가장 낮은 수준) + 23~24년 폐업률 증가하는 추세.

<개업/폐업 인사이트>
24년 개업률 감소 + 폐업률 증가 -> 강남, 용산,송파,은평 (순성장률 하락 구조)
대부분 자치구에서 2021~2022년 개업률 감소 이후 2023년 일부 회복이 나타났으나 2024년 다시 감소 흐름이 나타났고, 동시에 폐업률은 2023년 이후 증가하는 흐름

In [ ]:
import seaborn as sns

pivot_growth = growth.pivot(
    index='자치구_코드_명',
    columns='연도',
    values='순성장률'
)

plt.figure(figsize=(8,6))

sns.heatmap(
    pivot_growth,
    annot=True,
    fmt=".2f",
    cmap='RdBu_r',
    center=0
)

plt.title('선택구별 순성장률 Heatmap')
plt.show()

In [ ]:

import matplotlib.pyplot as plt
import platform



# 상권유형별 집계
구분통계 = 상권집계.groupby('상권_구분_코드_명').agg(
    매출건수=('총건수', 'sum'),
    평균매출=('총매출', 'mean'),
    중앙매출=('총매출', 'median'),
    평균객단가=('객단가', 'mean')
).reset_index()

구분통계['평균객단가'] = 구분통계['평균객단가'].round(0)

# 평균매출 기준 정렬
구분통계 = 구분통계.sort_values('평균매출', ascending=True)

fig, axes = plt.subplots(1, 4, figsize=(22, 10))

# 1. 매출건수
axes[0].barh(구분통계['상권_구분_코드_명'], 구분통계['매출건수'], color='steelblue')
axes[0].set_title('상권유형별 매출건수')
axes[0].set_xlabel('건수')


# 2. 평균매출
axes[1].barh(구분통계['상권_구분_코드_명'], 구분통계['평균매출'] / 1e8, color='coral')
axes[1].set_title('상권유형별 평균매출 (억원)')
axes[1].set_xlabel('억원')

# 3. 중앙매출
axes[2].barh(구분통계['상권_구분_코드_명'], 구분통계['중앙매출'] / 1e8, color='mediumpurple')
axes[2].set_title('상권유형별 중앙매출 (억원)')
axes[2].set_xlabel('억원')

# 4. 평균객단가
axes[3].barh(구분통계['상권_구분_코드_명'], 구분통계['평균객단가'], color='seagreen')
axes[3].set_title('상권유형별 평균객단가 (원)')
axes[3].set_xlabel('원')

plt.tight_layout()
plt.show()

# 전체연도

In [ ]:

import matplotlib.pyplot as plt
import platform

# 한글 폰트
if platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 자치구별 집계
전체_구분통계 = 전체_상권집계.groupby('자치구_코드_명').agg(
    매출건수=('총건수', 'sum'),
    평균매출=('총매출', 'mean'),
    중앙매출=('총매출', 'median'),
    평균객단가=('객단가', 'mean')
).reset_index()

전체_구분통계['평균객단가'] = 전체_구분통계['평균객단가'].round(0)

# 평균매출 기준 정렬
전체_구분통계 = 전체_구분통계.sort_values('평균매출', ascending=True)

fig, axes = plt.subplots(1, 4, figsize=(22, 10))

# 1. 매출건수
axes[0].barh(전체_구분통계['자치구_코드_명'], 전체_구분통계['매출건수'], color='steelblue')
axes[0].set_title('자치구별 매출건수')
axes[0].set_xlabel('건수')

# 2. 평균매출
axes[1].barh(전체_구분통계['자치구_코드_명'], 전체_구분통계['평균매출'] / 1e8, color='coral')
axes[1].set_title('자치구별 평균매출 (억원)')
axes[1].set_xlabel('억원')

# 3. 중앙매출
axes[2].barh(전체_구분통계['자치구_코드_명'], 전체_구분통계['중앙매출'] / 1e8, color='mediumpurple')
axes[2].set_title('자치구별 중앙매출 (억원)')
axes[2].set_xlabel('억원')

# 4. 평균객단가
axes[3].barh(전체_구분통계['자치구_코드_명'], 전체_구분통계['평균객단가'], color='seagreen')
axes[3].set_title('자치구별 평균객단가 (원)')
axes[3].set_xlabel('원')

plt.tight_layout()
plt.show()

In [ ]:

import matplotlib.pyplot as plt
import platform

# 한글 폰트
if platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 상권유형별 집계
전체_구분통계 = 전체_상권집계.groupby('상권_구분_코드_명').agg(
    매출건수=('총건수', 'sum'),
    평균매출=('총매출', 'mean'),
    중앙매출=('총매출', 'median'),
    평균객단가=('객단가', 'mean')
).reset_index()

전체_구분통계['평균객단가'] = 전체_구분통계['평균객단가'].round(0)

# 평균매출 기준 정렬
전체_구분통계 = 전체_구분통계.sort_values('평균매출', ascending=True)

fig, axes = plt.subplots(1, 4, figsize=(22, 10))

# 1. 매출건수
axes[0].barh(전체_구분통계['상권_구분_코드_명'], 전체_구분통계['매출건수'], color='steelblue')
axes[0].set_title('상권유형별 매출건수')
axes[0].set_xlabel('건수')


# 2. 평균매출
axes[1].barh(전체_구분통계['상권_구분_코드_명'], 전체_구분통계['평균매출'] / 1e8, color='coral')
axes[1].set_title('상권유형별 평균매출 (억원)')
axes[1].set_xlabel('억원')

# 3. 중앙매출
axes[2].barh(전체_구분통계['상권_구분_코드_명'], 전체_구분통계['중앙매출'] / 1e8, color='mediumpurple')
axes[2].set_title('상권유형별 중앙매출 (억원)')
axes[2].set_xlabel('억원')

# 4. 평균객단가
axes[3].barh(전체_구분통계['상권_구분_코드_명'], 전체_구분통계['평균객단가'], color='seagreen')
axes[3].set_title('상권유형별 평균객단가 (원)')
axes[3].set_xlabel('원')

plt.tight_layout()
plt.show()

In [ ]:
# 성별 비중
성별컬럼 = ['남성_매출_금액', '여성_매출_금액']
성별라벨 = ['남성', '여성']

gender_data = 매출_영역.groupby('자치구_코드_명')[성별컬럼].sum()
gender_data.columns = 성별라벨
gender_pct = gender_data.div(gender_data.sum(axis=1), axis=0) * 100

선택구 = ['서초구','강남구','용산구','중구','송파구','노원구','도봉구','은평구']

print(gender_pct.loc[선택구].round(1))

In [ ]:
# 성별 매출 비중
성별컬럼 = ['남성_매출_금액', '여성_매출_금액']
성별라벨 = ['남성', '여성']

선택구 = ['서초구','강남구','용산구','중구','송파구','노원구','도봉구','은평구']

gender_data = 매출_영역.groupby('자치구_코드_명')[성별컬럼].sum()
gender_data.columns = 성별라벨
gender_pct = gender_data.div(gender_data.sum(axis=1), axis=0) * 100
gender_pct = gender_pct.loc[선택구]

# 연령대 매출 비중
연령컬럼 = ['연령대_10_매출_금액','연령대_20_매출_금액','연령대_30_매출_금액',
        '연령대_40_매출_금액','연령대_50_매출_금액','연령대_60_이상_매출_금액']
연령라벨 = ['10대','20대','30대','40대','50대','60대+']

age_data = 매출_영역.groupby('자치구_코드_명')[연령컬럼].sum()
age_data.columns = 연령라벨
age_pct = age_data.div(age_data.sum(axis=1), axis=0) * 100
age_pct = age_pct.loc[선택구]

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# 성별
gender_pct.plot(kind='bar', stacked=True, ax=axes[0],
                color=['steelblue','coral'], width=0.7, edgecolor='white')
axes[0].set_title('자치구별 성별 매출 비중 (%)', fontweight='bold')
axes[0].set_ylabel('%')
axes[0].set_xlabel('')
axes[0].legend(title='성별', bbox_to_anchor=(1.02, 1), loc='upper left')
axes[0].tick_params(axis='x', rotation=0)

# 연령대
age_pct.plot(kind='bar', stacked=True, ax=axes[1],
             colormap='Set2', width=0.7, edgecolor='white')
axes[1].set_title('자치구별 연령대 매출 비중 (%)', fontweight='bold')
axes[1].set_ylabel('%')
axes[1].set_xlabel('')
axes[1].legend(title='연령대', bbox_to_anchor=(1.02, 1), loc='upper left')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# 성별 컬럼
성별컬럼 = ['남성_매출_금액', '여성_매출_금액']
성별라벨 = ['남성', '여성']

# 연도 생성 
매출_영역['연도'] = 매출_영역['기준_년분기_코드'] // 10

# 연도 + 자치구 기준 집계
gender_year = 매출_영역.groupby(
    ['연도','자치구_코드_명']
)[성별컬럼].sum()

# 컬럼 이름 변경
gender_year.columns = 성별라벨

# 비율 계산
gender_year_pct = gender_year.div(
    gender_year.sum(axis=1), axis=0
) * 100

# 선택구만 보기
선택구 = ['서초구','강남구','용산구','중구','송파구','노원구','도봉구','은평구']

print(gender_year_pct.loc[
    gender_year_pct.index.get_level_values('자치구_코드_명').isin(선택구)
].round(1))

In [ ]:
import matplotlib.pyplot as plt

for 구 in 선택구:
    temp = gender_year_pct.loc[
        gender_year_pct.index.get_level_values('자치구_코드_명')==구
    ]

    plt.plot(temp.index.get_level_values('연도'),
             temp['남성'],
             marker='o',
             label=구)

plt.title('자치구별 연도별 남성 매출 비중 변화')
plt.xlabel('연도')
plt.ylabel('남성 매출 비중 (%)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# 선택 자치구
선택구 = ['서초구','강남구','용산구','중구','송파구','노원구','도봉구','은평구']

plt.figure(figsize=(10,6))

for 구 in 선택구:
    temp = gender_year_pct.loc[
        gender_year_pct.index.get_level_values('자치구_코드_명') == 구
    ]

    plt.plot(
        temp.index.get_level_values('연도'),
        temp['여성'],
        marker='o',
        label=구
    )

plt.title('자치구별 연도별 여성 매출 비중 변화')
plt.xlabel('연도')
plt.ylabel('여성 매출 비중 (%)')
plt.legend(title='자치구', bbox_to_anchor=(1.02,1))
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

연도별, 상권구분코드명, 자치구=용산구, 직장인구 , 남녀비율
연도별, 상권구분코드명, 자치구=용산구, 상주인구 , 남녀비율
연도별, 상권구분코드명, 자치구=용산구, 유동인구 , 남녀비율
연도별, 자치구=용산구, 성장률
연도별, 자치구=용산구, 상권축소비율

In [ ]:
상권_통합.columns

In [ ]:
# 직장인구_필요컬럼 + 매출_영역 merge
직장_필요컬럼 = [
    '상권_코드',
    '기준_년분기_코드',
    '총_직장_인구_수',
    '남성_직장_인구_수',
    '여성_직장_인구_수'
]

매출_직장_통합 = 매출_영역.merge(
    df_직장[직장_필요컬럼],
    on=['상권_코드', '기준_년분기_코드'],
    how='left'
)

상권 상태별 평균매출비교, 상권변화상태별 매출흐름 분석.

In [ ]:
용산구_직장 = 매출_직장_통합[
    매출_직장_통합['자치구_코드_명'] == '용산구'
]



result = 용산구_직장.groupby(
    ['연도','상권_구분_코드_명']
).agg(
    총직장인구=('총_직장_인구_수','mean'),
    남성직장인구=('남성_직장_인구_수','mean'),
    여성직장인구=('여성_직장_인구_수','mean')
).reset_index()

result['남성비율'] = (
    result['남성직장인구'] /
    (result['남성직장인구'] + result['여성직장인구'])
) * 100

result['여성비율'] = 100 - result['남성비율']

result.round(1)

In [ ]:
용산구_직장['상권_구분_코드_명'].value_counts()

In [ ]:
매출_직장_통합.groupby('자치구_코드_명')['총_직장_인구_수'].mean()

In [ ]:
매출_직장_통합.groupby(
    ['상권_구분_코드_명']
)[['남성_직장_인구_수','여성_직장_인구_수']].mean()

In [ ]:
용산구_직장.groupby(
    ['상권_구분_코드_명']
)[['남성_직장_인구_수','여성_직장_인구_수']].mean()

In [ ]:
송파구_직장 = 매출_직장_통합[
    매출_직장_통합['자치구_코드_명'] == '송파구'
]

송파구_직장.groupby(
    ['상권_구분_코드_명']
)[['남성_직장_인구_수','여성_직장_인구_수']].mean()

In [ ]:
도봉구_직장 = 매출_직장_통합[
    매출_직장_통합['자치구_코드_명'] == '도봉구'
]

도봉구_직장.groupby(
    ['상권_구분_코드_명']
)[['남성_직장_인구_수','여성_직장_인구_수']].mean()

In [ ]:
중구_직장 = 매출_직장_통합[
    매출_직장_통합['자치구_코드_명'] == '중구'
]

중구_직장.groupby(
    ['상권_구분_코드_명']
)[['남성_직장_인구_수','여성_직장_인구_수']].mean()

In [ ]:
강남구_직장 = 매출_직장_통합[
    매출_직장_통합['자치구_코드_명'] == '강남구'
]

강남구_직장.groupby(
    ['상권_구분_코드_명']
)[['남성_직장_인구_수','여성_직장_인구_수']].mean()

In [ ]:
서초구_직장 = 매출_직장_통합[
    매출_직장_통합['자치구_코드_명'] == '서초구'
]

서초구_직장.groupby(
    ['상권_구분_코드_명']
)[['남성_직장_인구_수','여성_직장_인구_수']].mean()

In [ ]:
노원구_직장 = 매출_직장_통합[
    매출_직장_통합['자치구_코드_명'] == '노원구'
]

노원구_직장.groupby(
    ['상권_구분_코드_명']
)[['남성_직장_인구_수','여성_직장_인구_수']].mean()

In [ ]:
은평구_직장 = 매출_직장_통합[
    매출_직장_통합['자치구_코드_명'] == '은평구'
]

은평구_직장.groupby(
    ['상권_구분_코드_명']
)[['남성_직장_인구_수','여성_직장_인구_수']].mean()

* 상주인구. 남성여성비율

'총_상주인구_수',
       '남성_상주인구_수', '여성_상주인구_수'

In [ ]:
df_상주.columns

In [ ]:
# 상주인구_필요컬럼 + 매출_영역 merge
상주_필요컬럼 = [
    '상권_코드',
    '기준_년분기_코드',
    '총_상주인구_수',
    '남성_상주인구_수',
    '여성_상주인구_수'
]

매출_상주_통합 = 매출_영역.merge(
    df_상주[상주_필요컬럼],
    on=['상권_코드', '기준_년분기_코드'],
    how='left'
)

In [ ]:
매출_상주_통합.head(1)

In [ ]:
매출_상주_통합['남성_상주비율'] = (
    매출_상주_통합['남성_상주인구_수'] /
    매출_상주_통합['총_상주인구_수']
)

In [ ]:
매출_상주_통합['여성_상주비율'] = (
    매출_상주_통합['여성_상주인구_수'] /
    매출_상주_통합['총_상주인구_수']
)

In [ ]:
매출_상주_통합.groupby(
    ['상권_구분_코드_명']
)[['남성_상주인구_수','여성_상주인구_수']].mean()

In [ ]:
용산구_상주 = 매출_상주_통합[
    매출_상주_통합['자치구_코드_명'] == '용산구'
]


In [ ]:
용산구_상주.groupby(
    ['상권_구분_코드_명']
)[['남성_상주비율','여성_상주비율']].mean()

In [ ]:
용산구_상주.groupby(
    '상권_구분_코드_명'
)[['총_상주인구_수','남성_상주인구_수','여성_상주인구_수']].sum()

In [ ]:
tmp = 용산구_상주.groupby(
    '상권_구분_코드_명'
)[['총_상주인구_수','남성_상주인구_수','여성_상주인구_수']].sum()

tmp['남성_상주비율'] = tmp['남성_상주인구_수'] / tmp['총_상주인구_수']
tmp['여성_상주비율'] = tmp['여성_상주인구_수'] / tmp['총_상주인구_수']

tmp

In [ ]:
용산구_상주.groupby(
    ['상권_구분_코드_명']
)[['남성_상주인구_수','여성_상주인구_수']].mean()

In [ ]:
송파구_상주 = 매출_상주_통합[
    매출_상주_통합['자치구_코드_명'] == '송파구'
]

송파구_상주.groupby(
    ['상권_구분_코드_명']
)[['남성_상주인구_수','여성_상주인구_수']].mean()

In [ ]:
은평구_상주 = 매출_상주_통합[
    매출_상주_통합['자치구_코드_명'] == '은평구'
]

은평구_상주.groupby(
    ['상권_구분_코드_명']
)[['남성_상주인구_수','여성_상주인구_수']].mean()

* 길단위(유동인구)

In [ ]:
df_길단위.columns

In [ ]:
# 유동인구_필요컬럼 + 매출_영역 merge
유동_필요컬럼 = [
    '상권_코드',
    '기준_년분기_코드',
    '총_유동인구_수',
    '남성_유동인구_수',
    '여성_유동인구_수'
]

매출_유동_통합 = 매출_영역.merge(
    df_길단위[유동_필요컬럼],
    on=['상권_코드', '기준_년분기_코드'],
    how='left'
)

In [ ]:
매출_유동_통합.groupby(
    ['상권_구분_코드_명']
)[['남성_유동인구_수','여성_유동인구_수']].mean()

용산구,송파구, 도봉구,중구, 강남구,서초구,노원구,은평구

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 보고 싶은 자치구
target_gu = ['용산구', '송파구', '도봉구', '중구', '강남구', '서초구', '노원구', '은평구']

# 필터링
df_plot = 매출_유동_통합[매출_유동_통합['자치구_코드_명'].isin(target_gu)].copy()

# 집계 방식 선택: 'sum' 또는 'mean'
agg_func = 'mean'   # 유동인구 "수" 비교면 sum 추천 / 평균값 보고 싶으면 'mean'

# 자치구 + 상권유형별 집계
grouped = (
    df_plot
    .groupby(['자치구_코드_명', '상권_구분_코드_명'])[['남성_유동인구_수', '여성_유동인구_수']]
    .agg(agg_func)
    .reset_index()
)

print(grouped)

In [ ]:
import math

market_order = ['골목상권', '발달상권', '전통시장', '관광특구']

# 상권유형 순서 정리
grouped['상권_구분_코드_명'] = pd.Categorical(
    grouped['상권_구분_코드_명'],
    categories=market_order,
    ordered=True
)
grouped = grouped.sort_values(['자치구_코드_명', '상권_구분_코드_명'])

# subplot 설정
n = len(target_gu)
cols = 2
rows = math.ceil(n / cols)

fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
axes = axes.flatten()

for i, gu in enumerate(target_gu):
    ax = axes[i]
    temp = grouped[grouped['자치구_코드_명'] == gu].copy()
    
    x = np.arange(len(temp))
    width = 0.35
    
    bars1 = ax.bar(x - width/2, temp['남성_유동인구_수'], width, label='남성')
    bars2 = ax.bar(x + width/2, temp['여성_유동인구_수'], width, label='여성')
    
    ax.set_title(gu, fontsize=12)
    ax.set_xticks(x)
    ax.set_xticklabels(temp['상권_구분_코드_명'], rotation=0)
    ax.set_ylabel(f'유동인구 ({agg_func})')
    ax.legend()
    
    # 수치 라벨 표시
    for bars in [bars1, bars2]:
        for bar in bars:
            h = bar.get_height()
            ax.text(
                bar.get_x() + bar.get_width()/2,
                h,
                f'{h:,.0f}',
                ha='center',
                va='bottom',
                fontsize=9
            )

# 남는 축 제거
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

* 관광특구 매출건수 많은데 매출은 많은 이유 분석.

In [ ]:
관광특구_매출 = 매출_영역[
    매출_상주_통합['상권_구분_코드_명'] == '관광특구'
]

In [ ]:
관광특구_매출.groupby('상권_구분_코드_명')[
    ['시간대_17~21_매출_금액','시간대_21~24_매출_금액']
].mean()

In [ ]:
import matplotlib.pyplot as plt

# 그룹 평균 계산
df_plot = 관광특구_매출.groupby('상권_구분_코드_명')[
    ['시간대_17~21_매출_금액','시간대_21~24_매출_금액']
].mean()

# 시각화
ax = df_plot.plot(kind='bar', figsize=(8,5))

# 숫자 표시
for container in ax.containers:
    ax.bar_label(container, fmt='%.0f', label_type='edge')

plt.title('관광특구 시간대별 평균 매출 비교 (17~21 vs 21~24)')
plt.ylabel('평균 매출 금액')
plt.xticks(rotation=0)
plt.legend(title='시간대')
plt.tight_layout()

plt.show()

In [ ]:
관광특구_매출.groupby('상권_구분_코드_명')[
    ['주중_매출_금액','주말_매출_금액']
].mean()

In [ ]:
import matplotlib.pyplot as plt

# 그룹 평균 계산
df_plot = 관광특구_매출.groupby('상권_구분_코드_명')[
    ['주중_매출_금액','주말_매출_금액']
].mean()

# 시각화
ax = df_plot.plot(kind='bar', figsize=(8,5))

# 막대 위 숫자 표시 (천단위 콤마)
for container in ax.containers:
    ax.bar_label(container, fmt='{:,.0f}')

plt.title('관광특구 주중 vs 주말 평균 매출 비교')
plt.ylabel('평균 매출 금액')
plt.xticks(rotation=0)
plt.legend(title='구분')
plt.tight_layout()

plt.show()

In [ ]:
관광특구_매출.groupby('상권_구분_코드_명')[
    ['남성_매출_금액','여성_매출_금액']
].sum()

In [ ]:
import matplotlib.pyplot as plt

# 그룹 합계 계산
df_plot = 관광특구_매출.groupby('상권_구분_코드_명')[
    ['남성_매출_금액','여성_매출_금액']
].sum()

# 시각화
ax = df_plot.plot(kind='bar', figsize=(8,5))

# 막대 위 숫자 표시 (천단위 콤마)
for container in ax.containers:
    ax.bar_label(container, fmt='{:,.0f}')

plt.title('관광특구 남성 vs 여성 매출 총합 비교')
plt.ylabel('매출 금액 (합계)')
plt.xticks(rotation=0)
plt.legend(title='성별')
plt.tight_layout()

plt.show()

In [ ]:
관광특구_매출.groupby('상권_구분_코드_명')[
    ['연령대_20_매출_금액',
     '연령대_30_매출_금액',
     '연령대_40_매출_금액']
].sum()

In [ ]:
import matplotlib.pyplot as plt

# 그룹 합계 계산
df_plot = 관광특구_매출.groupby('상권_구분_코드_명')[
    ['연령대_20_매출_금액',
     '연령대_30_매출_금액',
     '연령대_40_매출_금액']
].sum()

# 컬럼명 보기 좋게 변경 (선택)
df_plot.columns = ['20대', '30대', '40대']

# 시각화
ax = df_plot.plot(kind='bar', figsize=(8,5))

# 막대 위 숫자 표시 (천단위 콤마)
for container in ax.containers:
    ax.bar_label(container, fmt='{:,.0f}')

plt.title('관광특구 연령대별 매출 총합 비교 (20·30·40대)')
plt.ylabel('매출 금액 (합계)')
plt.xticks(rotation=0)
plt.legend(title='연령대')
plt.tight_layout()

plt.show()

In [ ]:
업종수=('서비스_업종_코드','nunique')

In [ ]:
매출_영역.groupby('상권_구분_코드_명')['서비스_업종_코드'].nunique()

In [ ]:
import matplotlib.pyplot as plt

# 상권 유형별 업종 수 계산
df_plot = 매출_영역.groupby('상권_구분_코드_명')['서비스_업종_코드'].nunique()

# 시각화
ax = df_plot.plot(kind='bar', figsize=(7,5))

# 막대 위 숫자 표시
for i, v in enumerate(df_plot):
    plt.text(i, v, f'{v:,}', ha='center', va='bottom')

plt.title('상권 유형별 서비스 업종 다양성 비교')
plt.ylabel('서비스 업종 개수')
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

① 용산구 상권유형 구성
② 상권유형별 남녀 직장인 비율
③ 용산구 vs 타구 비교
④ 연도별 변화 여부 확인

In [ ]:

매출_상주_통합.groupby('상권_구분_코드_명')['총_상주인구_수'].mean()

In [ ]:
매출_유동_통합.groupby('상권_구분_코드_명')['총_유동인구_수'].mean()

In [ ]:
매출_직장_통합.groupby('상권_구분_코드_명')['총_직장_인구_수'].mean()

In [ ]:
import matplotlib.pyplot as plt

# 상권 유형별 평균 상주인구 계산
df_plot = 매출_상주_통합.groupby('상권_구분_코드_명')['총_상주인구_수'].mean()

# 시각화
ax = df_plot.plot(kind='bar', figsize=(7,5))

# 막대 위 숫자 표시 (천단위 콤마)
for i, v in enumerate(df_plot):
    plt.text(i, v, f'{v:,.0f}', ha='center', va='bottom')

plt.title('상권 유형별 평균 상주인구 비교')
plt.ylabel('평균 상주인구 수')
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# 데이터 (이미 계산된 Series라 가정)
유동인구 = 매출_유동_통합.groupby('상권_구분_코드_명')['총_유동인구_수'].mean()

# 그래프
plt.figure(figsize=(8,5))

ax = 유동인구.plot(kind='bar')

plt.title('상권 유형별 평균 유동인구 비교')
plt.xlabel('상권 유형')
plt.ylabel('평균 유동인구 수')

# 숫자 표시
for i, v in enumerate(유동인구):
    plt.text(i, v, f'{int(v):,}', ha='center', va='bottom')

plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# 직장인구 데이터 (이미 계산된 Series라고 가정)
직장인구 = 매출_직장_통합.groupby('상권_구분_코드_명')['총_직장_인구_수'].mean()

# 큰 값부터 정렬
직장인구_sorted = 직장인구.sort_values(ascending=False)

# 그래프
plt.figure(figsize=(8,5))

ax = 직장인구_sorted.plot(kind='bar')

plt.title('상권 유형별 평균 직장인구 비교')
plt.xlabel('상권 유형')
plt.ylabel('평균 직장인구 수')

# 수치 표시
for i, v in enumerate(직장인구_sorted):
    plt.text(i, v, f'{int(v):,}', ha='center', va='bottom')

plt.xticks(rotation=0)
plt.tight_layout()
plt.show()